# IE 643 · Content vs. motion in AnimateDiff — the interface on a Kaggle GPU

**How to use (interactive session):**
1. *Settings → Accelerator → GPU T4 x2* (or P100) and *Settings → Internet → On*.
2. Check that the two datasets are attached (right panel → Input): `vdprobe-code` and `vdprobe-app-assets`.
3. *Run all*. The last cell prints a **public URL** (`https://….gradio.live`, valid ~72 h while the session runs):
   open it in any browser and use the app exactly as on a local GPU. Stop the session when done (it uses GPU quota).

Model weights (~5 GB) download on the first run (~3 min); the first edit loads the models (~1 min).
`--variant epicrealism_mm3` (etc.) in the last cell switches the model cell.

In [ ]:
import glob, os, shutil, subprocess, sys
def find(name):
    hits = [p for p in glob.glob('/kaggle/input/**/' + name, recursive=True) if os.path.isdir(p)]
    hits += [p for p in glob.glob('/kaggle/input/' + name)]
    if not hits:
        raise SystemExit(f'dataset {name} is not attached')
    return sorted(hits, key=len)[0]
CODE, ASSETS = find('vdprobe-code'), find('vdprobe-app-assets')
REPO = '/kaggle/working/repo'
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(CODE, REPO)
print('code', CODE, '\nassets', ASSETS)

In [ ]:
import torch
GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if GPU else 'none (enable a GPU accelerator to run the app)')

In [ ]:
if GPU:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-input', 'diffusers==0.40.0', 'peft==0.21.2',
                    'gradio==6.30.0', 'scikit-learn', 'opencv-python-headless'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'])   # breaks peft on this image

In [ ]:
# Runs until you interrupt it. Look for 'Running on public URL'.
if GPU:
    os.chdir(REPO)
    env = dict(os.environ, PYTHONPATH=REPO + '/src', PYTHONUNBUFFERED='1')
    p = subprocess.Popen([sys.executable, 'scripts/app.py', '--share', '--preload', '--assets', ASSETS],
                         env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
else:
    print('No GPU in this session: enable one (Settings -> Accelerator) and run again.')